In [2]:
import torch
import torch.nn as nn

# Параметры
batch_size = 2      # У нас 2 предложения
seq_len = 5         # В каждом предложении по 5 слов (токенов)
input_size = 32     # Размер вектора одного слова (embedding_dim)
hidden_size = 64    # Размер вектора "памяти"

# 1. Создаем слой RNN
# Обязательно указываем batch_first=True, чтобы PyTorch ждал тензоры 
# в привычном нам виде: [batch_size, seq_len, features]
rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)

# 2. Создаем фейковый батч предложений (векторы слов после слоя Embedding)
# Размер: [2 предложения, 5 слов, 32 числа на слово]
x = torch.randn(batch_size, seq_len, input_size)

# 3. Прогоняем данные через RNN
# RNN всегда возвращает два объекта: output и hidden
output, hidden = rnn(x)

print("Форма output:", output.shape) # torch.Size([2, 5, 64])
print("Форма hidden:", hidden.shape) # torch.Size([1, 2, 64])

Форма output: torch.Size([2, 5, 64])
Форма hidden: torch.Size([1, 2, 64])


In [4]:
# Матрица W_hx (Вход -> Память)
print(rnn.weight_ih_l0.shape)  # torch.Size([64, 32]) 

# Матрица W_hh (Прошлая память -> Новая память)
print(rnn.weight_hh_l0.shape)  # torch.Size([64, 64])

torch.Size([64, 32])
torch.Size([64, 64])


In [5]:
class TextRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        
        # 1. Слой превращения индексов слов в векторы
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)
        
        # 2. Рекуррентный слой (Читает слова по очереди и копит память)
        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_size, batch_first=True)
        
        # 3. Матрица W_hy (Память -> Выход классификатора)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x размер: [batch_size, seq_len] (например, [2, 5])
        
        # 1. Получаем векторы слов
        emb = self.embedding(x) 
        # emb размер: [batch_size, seq_len, embed_dim] -> [2, 5, 32]
        
        # 2. Прогоняем через RNN
        # Нас интересует только 'hidden' - финальный смысл всего предложения
        out, hidden = self.rnn(emb) 
        # hidden размер: [1, batch_size, hidden_size] -> [1, 2, 64]
        
        # 3. Убираем лишнее первое измерение (1 слой), чтобы передать в Linear
        final_memory = hidden.squeeze(0) 
        # final_memory размер: [batch_size, hidden_size] -> [2, 64]
        
        # 4. Классифицируем
        logits = self.fc(final_memory)
        # logits размер: [batch_size, num_classes] -> [2, 3]
        
        return logits

# Проверяем работу сети
model = TextRNNClassifier(vocab_size=1000, embed_dim=32, hidden_size=64, num_classes=3)
dummy_indices = torch.randint(1, 1000, (2, 5)) # 2 предложения по 5 слов
predictions = model(dummy_indices)

print("Финальные предсказания:", predictions.shape) # [2, 3]

Финальные предсказания: torch.Size([2, 3])
